# ATLAS Z Peak Demo

This notebook demonstrates the project workflow using `atlasopenmagic` to find ATLAS Open Data URLs, then running the reusable pipeline for `Z -> mumu`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from atlas_analysis.data_access import get_urls, write_url_file
from atlas_analysis.pipeline import AnalysisConfig, run_analysis
from atlas_analysis.physics import ObjectCuts
from atlas_analysis.schema import describe_inspection

## Get One Real ATLAS Data URL

The 2020 13 TeV education release has an `exactly2lep` skim that is ideal for reconstructing the Z peak.

In [ ]:
urls = get_urls(release="2020e-13tev", dataset="data", skim="exactly2lep", protocol="https", limit=1)
write_url_file(urls, "data/atlas_urls.txt")
urls[0]

In [ ]:
print(describe_inspection(urls[0]))

## Run `Z -> mumu`

In [ ]:
result = run_analysis(
    AnalysisConfig(
        input_files=urls,
        output_folder="outputs",
        max_events=10_000,
        channel="z_mumu",
        cuts=ObjectCuts(z_mass_min=80, z_mass_max=100),
    )
)
result.summary

In [ ]:
pd.read_csv("outputs/cutflow.csv")

## Show the Z Peak

In [ ]:
z = pd.read_csv("outputs/z_mass_spectrum.csv")
zmumu = z[z["channel"] == "Z->mumu"]
plt.figure(figsize=(8, 5))
plt.step(zmumu["bin_center"], zmumu["raw_count"], where="mid")
plt.axvline(91.1876, color="tab:red", linestyle="--", label="Z mass")
plt.xlabel("m_mumu [GeV]")
plt.ylabel("Events")
plt.legend()
plt.grid(alpha=0.25)
plt.show()